# NB09 — Musanga ecology: disturbance legacy and spectral recovery
V5 supplementary ecological analysis, not another mapping model search. CPU, cache-only, run all cells. Requires corrected NB08bis Block 14B outputs. No Earth Engine downloads, no production modifications. Hypotheses: (1) abundance differs with detected disturbance regime and age; (2) pre-survey spectral trajectories differ with abundance; (3) HLS changes align with detected disturbance events. All associations observational, four landscapes only. Hansen no-loss is not proof of undisturbed forest; JRC is not independent truth. Equal-weight forest summaries and within-site 500m block uncertainty, not independent pixels. Use the full reference cohort for disturbance analyses and matched annual observations for HLS analyses. Avoid interpreting disagreement as error without independent imagery. 


In [ ]:
%pip -q install rasterio geopandas pyarrow scikit-learn scipy pandas matplotlib requests joblib
from pathlib import Path
from contextlib import ExitStack
from datetime import datetime,timezone
from collections import defaultdict
import os,json,hashlib,time,math,requests
import numpy as np,pandas as pd,geopandas as gpd,matplotlib.pyplot as plt
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.transform import from_origin,Affine
from rasterio.io import MemoryFile
from rasterio.warp import transform_bounds
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score,average_precision_score
from scipy.stats import spearmanr
from threadpoolctl import threadpool_limits
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
threadpool_limits(limits=1)
V5_ROOT=next((p for p in (Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/V5'),Path('/content/drive/MyDrive/Musanga/Final/V5')) if (p.parent/'Data').is_dir() and p.is_dir()),None)
assert V5_ROOT is not None
ROOT=V5_ROOT/'Outputs/08_satellite_reference';ROOT.mkdir(parents=True,exist_ok=True)
OUT=V5_ROOT/'Outputs/08_satellite_science_v5';OUT.mkdir(parents=True,exist_ok=True)
V4_OUT=OUT # compatibility variable for retained analysis functions; never a V4 path
NB07=V5_ROOT/'Outputs/07_RGB_unified_production_v5'
MANIFEST=NB07/'07_production_mosaic_manifest.json'
assert MANIFEST.is_file(),'NB07 must finish all six mosaics before NB08 execution.'
M=json.loads(MANIFEST.read_text())
assert M['complete'] and M['schema']=='musanga_v5_nb07_rgb_mosaics' and len(M['mosaics'])==6
SITES=('Baego','MOS','Yangambi','Yoko')
YEARS={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
KEYS=['site','mosaic','row','col']
CRS='EPSG:32635';TILE_M=1500;LATTICE_M=1;MIN_SUPPORT=.90
SCALES=[10,25,30,50,75,100]
EE_PROJECT='musanga-511110' # change to an authorized project when transferring
S2_SOURCE='COPERNICUS/S2_SR_HARMONIZED'
EMB_SOURCE='GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'
S2_BANDS=['B2','B3','B4','B8','B5','B6','B7','B8A','B11','B12']
SCL_REJECT=[0,1,3,7,8,9,10,11]
INDEX_NAMES=['NDVI','NDMI','NBR','NDRE']
EMB_BANDS=[f'A{i:02d}' for i in range(64)]
ANNUAL=[f'RAW_ANNUAL__{i:02d}' for i in range(1,16)]
QUARTER=[f'S2_QUARTERLY_3YR__{i:02d}' for i in range(1,45)]
EMB=[f'EMB_ANNUAL__{i:02d}' for i in range(1,65)]
Q_COUNTS=[QUARTER[i-1] for i in (11,22,33,44)]
S2=ANNUAL+[c for c in QUARTER if c not in Q_COUNTS]
FEATURE_SETS={'M1_S2':S2,'M2_EMB':EMB,'M3_S2_EMB':S2+EMB}
ALL_FEATURES=sorted(set(sum(FEATURE_SETS.values(),[])))
SEED=20261120;SEED17=SEED;CAP_PER_TRAIN_FOREST=1500
MODEL=dict(learning_rate=.06,max_iter=180,max_leaf_nodes=20,min_samples_leaf=35,l2_regularization=2,early_stopping=False)
THRESHOLDS=(.05,.20,.40)
RUN_STRUCTURE=True  # supplementary; independent GEE assets may require access
RUN_ABUNDANCE_SENSITIVITY=True
# Normal restart: reuse completed data and analyses. True rebuilds derived aggregates/results;
# validated per-tile imagery and UAV tiles remain reusable.
FORCE_REBUILD=False
def digest(obj):return hashlib.sha256(json.dumps(obj,sort_keys=True,default=str).encode()).hexdigest()
def atomic_json(p,obj):
 p=Path(p);tmp=p.with_suffix(p.suffix+'.partial');tmp.write_text(json.dumps(obj,indent=2,default=str));os.replace(tmp,p)
def atomic_parquet(d,p):
 p=Path(p);tmp=p.with_suffix('.partial');d.to_parquet(tmp,index=False);os.replace(tmp,p)
def dataframe_sha(d):return hashlib.sha256(pd.util.hash_pandas_object(d,index=False).values.tobytes()).hexdigest()

from scipy.spatial import cKDTree
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn import __version__ as SK_VERSION
from scipy.ndimage import uniform_filter
NB08_OUT=OUT
OUT=V5_ROOT/'Outputs/08bis_abundance_exploration_v5';OUT.mkdir(exist_ok=True)
RUN_SCALE=True;RUN_CONTEXT=True;RUN_NICFI=True;RUN_HLS=True
BASE=S2+EMB;BASE_NAME='M3_S2_EMB__HGB__forest_equal'
candidates=[]
for p in sorted(ROOT.glob('native_50m_reference_*.parquet')):
 sig=p.stem.removeprefix('native_50m_reference_');cf=NB08_OUT/sig/'config.json'
 if not cf.is_file():continue
 c=json.loads(cf.read_text());tc=ROOT/'uav_targets'/c['target_id']/'config.json'
 if not tc.is_file() or digest(c)[:20]!=sig or c['features']!=FEATURE_SETS:continue
 t=json.loads(tc.read_text())
 if t['model']!=M['model_sha'] or t['policy']!=M['policy_sha'] or t['support']!=MIN_SUPPORT:continue
 old={a['name']:a for a in t['mosaics']}
 if set(old)!={a['name'] for a in M['mosaics']}:continue
 valid=True
 for a in M['mosaics']:
  f=NB07/'reconstructed_mosaics'/Path(a['output']).name
  if not f.is_file() or old[a['name']]['digest']!=a['digest'] or old[a['name']]['size']!=f.stat().st_size:valid=False;break
  with rasterio.open(f) as ds:
   if ds.tags().get('signature')!=a['digest'] or list(ds.bounds)!=old[a['name']]['bounds']:valid=False;break
 if not valid:continue
 d=pd.read_parquet(p)
 if dataframe_sha(d[KEYS+['x','y','target_fraction']+ALL_FEATURES])==c['source_sha256']:candidates.append((p,c,d))
assert candidates,'Complete NB08 reference first'
assert len({c['source_sha256'] for _,c,_ in candidates})==1,'Multiple distinct references; select explicitly'
REF,CONFIG,D=candidates[0];SOURCE_SHA=CONFIG['source_sha256']
SAT_ID=CONFIG['satellite_id'];SAT_DIR=ROOT/'satellite_10m'/SAT_ID
TARGET_DIR=ROOT/'uav_targets'/CONFIG['target_id']
assert len(D)==CONFIG['n'] and set(D.site)==set(SITES) and not D.duplicated(['site','row','col']).any()
assert D.target_fraction.between(0,1).all() and np.isfinite(D[BASE].to_numpy(float)).all()
found=[]
for p in sorted((NB08_OUT/'within_landscape_50m').glob('*/heldout_predictions.parquet')):
 c=json.loads((p.parent/'config.json').read_text());z=pd.read_parquet(p)
 if c.get('version')!='within_landscape_v2' or c.get('buffer')!=100 or c.get('block')!=500 or BASE_NAME not in z:continue
 cols=['site','row','col','x','y','target_fraction']
 if not z[cols].equals(D[cols]):continue
 used=sorted(set(sum(c['features'].values(),[])))
 if dataframe_sha(D.assign(fold21=z.fold21.to_numpy())[['site','x','y','target_fraction','fold21']+used])==c['source']:found.append((p,z))
assert found,'Complete NB08 spatial holdout first'
BASE_P,OLD=found[0];M21=D.assign(block21=OLD.block21.to_numpy(),fold21=OLD.fold21.to_numpy())
mapping=M21.groupby('block21').fold21.first().to_dict()
assert M21.groupby('block21').fold21.nunique().eq(1).all() and M21.groupby('site').fold21.nunique().eq(5).all()
RUN=OUT/digest(dict(version='08bis_v1',source=SOURCE_SHA,fold=dataframe_sha(M21[['site','row','col','fold21']]),sklearn=SK_VERSION))[:20];RUN.mkdir(exist_ok=True)
atomic_json(RUN/'input_contract.json',dict(reference=str(REF),source=SOURCE_SHA,nb08_predictions=str(BASE_P),satellite=SAT_ID,sklearn=SK_VERSION))
REPORTS=[];SEED21=20261123
print('Reference:',REF,'\nPredictions:',BASE_P,'\nOutput:',RUN)
display(M21.groupby(['site','fold21']).size().unstack())


## 01 — Verified corrected history inputs, full reference cohort and QA

In [ ]:
# Locate corrected cache by reference/fold identity, not by latest modification.
foldsha=dataframe_sha(M21[['site','row','col','fold21']])
candidates=[]
for p in OUT.glob('*/14_long_history/*/hansen_mask_corrected_v2/features_50m.parquet'):
 c=json.loads((p.parent/'config.json').read_text())
 if c['parent']['source']==SOURCE_SHA and c['parent']['fold']==foldsha:
  a=p.parent.parent/'annual_HLS_50m.parquet'
  if a.is_file():candidates.append((p,a,c))
assert len(candidates)==1,('Need exactly one corrected matching Block 14 cache',candidates)
fp,ap,contract=candidates[0];F=pd.read_parquet(fp);A=pd.read_parquet(ap)
assert len(F)==len(M21) and not F.duplicated(KEYS).any()
assert not A.duplicated(KEYS+['year']).any()
D9=M21.merge(F,on=KEYS,validate='one_to_one')
assert len(D9)==len(M21)
assert D9.groupby('site').QA14_hansen_support.mean().ge(.8).all()
assert (A.year<A.site.map(YEARS)).all()
assert set(A.site)==set(SITES)
A=A.drop(columns=['target_fraction','x','y'],errors='ignore').merge(D9[KEYS+['target_fraction','block21','eligible14']],on=KEYS,validate='many_to_one')
O9=V5_ROOT/'Outputs/09_ecology_v5'/digest(dict(version='ecology_v1',source=SOURCE_SHA,features=dataframe_sha(F),annual=dataframe_sha(A)))[:20];O9.mkdir(parents=True,exist_ok=True)
atomic_json(O9/'input_contract.json',dict(reference=str(REF),features=str(fp),annual=str(ap),source=SOURCE_SHA,contract=contract))
D9['cover_pp']=100*D9.target_fraction
D9['rich']=D9.target_fraction>=.4
D9['history_ok']=D9.QA14_hansen_support.ge(.9)&D9.QA14_jrc_support.ge(.9)
Q=D9[D9.history_ok].copy()
print('Full reference / history valid / HLS matched',len(D9),len(Q),int(D9.eligible14.sum()))
display(D9.groupby('site').agg(n=('rich','size'),history_valid=('history_ok','sum'),HLS_matched=('eligible14','sum')))
SEED9=20261126;BOOT9=1000
def save9(df,name):
 df.to_csv(O9/(name+'.csv'),index=False);return df
def fig9(fig,name):
 fig.savefig(O9/(name+'.png'),dpi=200,bbox_inches='tight');plt.show();plt.close(fig)
def group_ci(g,col):
 # Cluster bootstrap within site; conditional descriptive uncertainty.
 q=g.groupby('block21')[col].agg(['sum','count']).to_numpy(float)
 if len(q)<5:return (np.nan,np.nan,len(q))
 rng=np.random.default_rng(SEED9);v=q[rng.integers(0,len(q),(BOOT9,len(q)))].sum(1);z=v[:,0]/v[:,1]
 return (*np.quantile(z,[.025,.975]),len(q))
def summary9(g):
 lo,hi,nb=group_ci(g,'cover_pp')
 return dict(n=len(g),blocks=nb,mean_cover=g.cover_pp.mean(),median_cover=g.cover_pp.median(),q25=g.cover_pp.quantile(.25),q75=g.cover_pp.quantile(.75),rich_fraction=g.rich.mean(),ci_low=lo,ci_high=hi)


## 02 — Disturbance regime, agreement and abundance. Cell-level fractions, not pixel agreement. Primary threshold 20% affected cell area; sensitivity any detected and ≥50%. No-loss ≠ undisturbed. JRC includes historical degradation/deforestation states; regrowth is reported separately.

In [ ]:
reg=[]
for threshold in [.000001,.2,.5]:
 q=Q.copy();h=q.G14_loss_any>=threshold;j=q.J14_ever_disturbed>=threshold
 q['regime']=np.select([~h&~j,~h&j,h&~j,h&j],['Neither detected','JRC only','Hansen only','Both detected'],default='unknown')
 for (site,regime),g in q.groupby(['site','regime']):
  reg.append(dict(threshold=threshold,site=site,regime=regime,**summary9(g)))
 if threshold==.2:Q['regime']=q.regime
R=save9(pd.DataFrame(reg),'disturbance_regime_cover')
display(R[R.threshold.eq(.2)].round(3))
order=['Neither detected','JRC only','Hansen only','Both detected']
fig,axs=plt.subplots(2,2,figsize=(12,8),layout='constrained')
for s,ax in zip(SITES,axs.ravel()):
 g=R[(R.site==s)&R.threshold.eq(.2)].set_index('regime').reindex(order)
 x=np.arange(4);ax.bar(x,g.mean_cover,color='steelblue');ax.errorbar(x,g.mean_cover,yerr=[(g.mean_cover-g.ci_low).clip(lower=0),(g.ci_high-g.mean_cover).clip(lower=0)],fmt='none',color='black')
 ax.set_xticks(x,[f'{k}\nn={int(v) if np.isfinite(v) else 0}' for k,v in zip(order,g.n)],rotation=15);ax.set(title=s,ylabel='Mean Musanga cover (%)',ylim=(0,100))
fig9(fig,'disturbance_regime_cover')
# Within-site contrasts vs neither, spatial blocks jointly resampled.
contr=[]
for s,g in Q.groupby('site'):
 for label in order[1:]:
  a=g[g.regime.isin([order[0],label])]
  tab=a.groupby(['block21','regime']).cover_pp.agg(['sum','count']).unstack(fill_value=0)
  if not all((k,l) in tab for k in ['sum','count'] for l in [order[0],label]):continue
  x=tab[[('sum',order[0]),('count',order[0]),('sum',label),('count',label)]].to_numpy(float)
  if min((x[:,1]>0).sum(),(x[:,3]>0).sum())<5:continue
  rng=np.random.default_rng(SEED9);z=x[rng.integers(0,len(x),(BOOT9,len(x)))].sum(1);z=z[(z[:,1]>0)&(z[:,3]>0)]
  d=z[:,2]/z[:,3]-z[:,0]/z[:,1];t=x.sum(0)
  contr.append(dict(site=s,regime=label,delta_pp=t[2]/t[3]-t[0]/t[1],ci_low=np.quantile(d,.025),ci_high=np.quantile(d,.975),blocks=len(x)))
display(save9(pd.DataFrame(contr),'regime_paired_block_contrasts').round(3))


## 03 — Successional window, disturbance intensity and recurrence. Conditional age of detected loss, not stand age. No detected loss separate; mixed-age cells use loss-weighted mean. JRC duration is consecutive observed regrowth state, not actual stand age.

In [ ]:
age=[]
for name,frac,agecol in [('Hansen','G14_loss_any','G14_loss_age_conditional'),('JRC regrowth','J14_last_regrowth','J14_regrowth_age_conditional')]:
 for th in [.2,.5]:
  q=Q[Q[frac]>=th].copy()
  q['age_bin']=pd.cut(q[agecol],[0,3,7,12,20,40],include_lowest=True)
  for (s,b),g in q.groupby(['site','age_bin'],observed=True):
   age.append(dict(source=name,threshold=th,site=s,age_bin=str(b),**summary9(g)))
AGE=save9(pd.DataFrame(age),'successional_age_cover')
display(AGE.round(3))
fig,axs=plt.subplots(2,4,figsize=(17,8),layout='constrained')
for i,name in enumerate(['Hansen','JRC regrowth']):
 for j,s in enumerate(SITES):
  ax=axs[i,j];g=AGE[(AGE.source==name)&AGE.threshold.eq(.2)&AGE.site.eq(s)]
  x=np.arange(len(g));ax.plot(x,g.mean_cover,'o-');ax.fill_between(x,g.ci_low,g.ci_high,alpha=.18);ax.set_xticks(x,[f'{b}\nn={n}' for b,n in zip(g.age_bin,g.n)],rotation=40);ax.set(title=f'{s}: {name}',ylim=(0,100),ylabel='Mean cover (%)')
fig9(fig,'successional_age_cover')
intensity=[]
for col in ['G14_loss_any','J14_ever_disturbed','J14_ever_regrowth','J14_last_degraded','J14_last_regrowth','J14_disturbance_entries']:
 q=Q.copy();edges=[-1e-9,1e-8,.2,.5,1.00001] if col!='J14_disturbance_entries' else [-1e-9,1e-8,1,2,50]
 q['category']=pd.cut(q[col],edges,include_lowest=True)
 for (s,b),g in q.groupby(['site','category'],observed=True):intensity.append(dict(feature=col,site=s,category=str(b),**summary9(g)))
save9(pd.DataFrame(intensity),'disturbance_intensity_recurrence')


## 04 — HLS trajectories by Musanga abundance and disturbance regime. Use common relative lags (1–7) for cross-site comparison; longer records retained for site-specific plots. Annual available-case summaries with counts. Within-site associations to avoid conflating site and sensor era.

In [ ]:
BANDS=['NDVI','NDMI','NBR','NIR','SWIR1','SWIR2']
A=A.merge(Q[KEYS+['regime']],on=KEYS,how='left',validate='many_to_one')
A['cover_class']=pd.cut(A.target_fraction,[-.001,.05,.2,.4,1.001],right=False,labels=['<5%','5–20%','20–40%','≥40%'])
tra=[]
for band in BANDS:
 for (s,c,lag),g in A.groupby(['site','cover_class','lag'],observed=True):
  v=g.dropna(subset=[band])
  if len(v):tra.append(dict(band=band,site=s,cover_class=str(c),lag=lag,n=len(v),blocks=v.block21.nunique(),median=v[band].median(),q25=v[band].quantile(.25),q75=v[band].quantile(.75)))
TR=save9(pd.DataFrame(tra),'annual_HLS_by_abundance')
for band in BANDS:
 fig,axs=plt.subplots(2,2,figsize=(11,8),layout='constrained')
 for s,ax in zip(SITES,axs.ravel()):
  for c,g in TR[(TR.band==band)&TR.site.eq(s)].groupby('cover_class'):
   g=g.sort_values('lag');ax.plot(-g.lag,g['median'],'o-',label=c);ax.fill_between(-g.lag,g.q25,g.q75,alpha=.12)
  ax.set(title=s,xlabel='Years before survey',ylabel=band);ax.legend(fontsize=8)
 fig9(fig,'HLS_trajectory_'+band)
# Spearman associations by forest, both full history and recent 3 years; descriptive.
assoc=[]
for s,g in D9.groupby('site'):
 for col in [c for c in D9 if c.startswith(('H14_','G14_','J14_'))]:
  v=g[['target_fraction',col]].dropna()
  if len(v)>10 and v[col].nunique()>1 and v.target_fraction.nunique()>1:
   assoc.append(dict(site=s,feature=col,n=len(v),rho=spearmanr(v.target_fraction,v[col]).statistic))
AS=save9(pd.DataFrame(assoc),'within_forest_feature_associations')
display(AS[AS.feature.isin(['G14_loss_any','G14_loss_age_conditional','J14_ever_disturbed','J14_regrowth_age_conditional','H14_NBR_trend','H14_NDMI_trend'])].round(3))


## 05 — Event-aligned HLS vs Hansen and JRC. Read corrected cached 30 m history rasters, derive dominant event year and affected area at 50 m. Hansen loss year and first/last JRC entry into degraded/deforested state. Require ≥50% cell affected by one event year for primary alignment; ≥20% sensitivity. Exclude events without adequate pre/post HLS support. Compare spectral change relative to pre-event mean (years −2,−1); event year 0 may mix pre/post. JRC transitions are state-change proxies; overlapping events and Landsat shared inputs prevent independent validation.

In [ ]:
def event_tile(g,site,tx,ty):
 path=fp.parent/f'{site}_{tx}_{ty}_history.tif'
 assert path.is_file(),path
 with rasterio.open(path) as ds:
  a=ds.read();names=list(ds.descriptions)
  assert ds.shape==(50,50) and names[:2]==['treecover2000','lossyear']
  assert names[2:]==[f'JRC_{k}' for k in range(1990,YEARS[site])]
  assert ds.crs==rasterio.crs.CRS.from_string(CRS)
 loss=a[1];j=a[2:];valid=(j>=1)&(j<=6);dis=np.isin(j,[2,3]);entry=dis[1:]&~dis[:-1]&valid[1:]&valid[:-1]
 years=np.arange(1991,YEARS[site])[:,None,None]
 first=np.min(np.where(entry,years,9999),axis=0);last=np.max(np.where(entry,years,0),axis=0)
 first=np.where(first==9999,0,first)
 # Native 30m raster to aligned 10m subcells; cell-specific area fraction.
 rr=np.floor(((ty+1)*1500-g.y.to_numpy())/50).astype(int);cc=np.floor((g.x.to_numpy()-tx*1500)/50).astype(int)
 def area(mask):
  z=np.repeat(np.repeat(mask,3,0),3,1).reshape(30,5,30,5).sum((1,3))/25
  return z[rr,cc]
 out=g[KEYS].reset_index(drop=True).copy()
 for tag,yr in [('Hansen',np.where(loss>0,2000+loss,0)),('JRC_first',first),('JRC_last',last)]:
  eventyears=np.arange(1990,YEARS[site]);fr=np.stack([area(yr==k) for k in eventyears],1)
  k=fr.argmax(1);v=fr[np.arange(len(g)),k]
  out[tag+'_year']=np.where(v>0,eventyears[k],np.nan);out[tag+'_fraction']=v
 return out
ev=[]
for (s,tx,ty),g in Q.groupby(['site','tile_x','tile_y']):
 ev.append(event_tile(g,s,int(tx),int(ty)))
E=pd.concat(ev,ignore_index=True);assert len(E)==len(Q) and not E.duplicated(KEYS).any();atomic_parquet(E,O9/'event_years_50m.parquet')
# Compare timing only where each product detects a spatially dominant event.
Z=Q.merge(E,on=KEYS,validate='one_to_one');agreement=[]
for s,g in Z.groupby('site'):
 for th in [.2,.5]:
  for jtag in ['JRC_first','JRC_last']:
   z=g[(g.Hansen_fraction>=th)&(g[jtag+'_fraction']>=th)]
   delta=z[jtag+'_year']-z.Hansen_year
   agreement.append(dict(site=s,threshold=th,jrc_event=jtag,n=len(z),median_difference=delta.median(),within_1yr=(delta.abs()<=1).mean(),within_2yr=(delta.abs()<=2).mean()))
display(save9(pd.DataFrame(agreement),'event_year_agreement').round(3))
event_rows=[];event_changes=[]
for tag in ['Hansen','JRC_first','JRC_last']:
 for th in [.2,.5]:
  z=Z[Z[tag+'_fraction']>=th][KEYS+[tag+'_year','cover_pp','rich','block21']]
  a=A.drop(columns=['block21'],errors='ignore').merge(z,on=KEYS,validate='many_to_one')
  a['rel']=a.year-a[tag+'_year']
  for band in ['NBR','NDMI','NDVI']:
   pre=a[a.rel.isin([-2,-1])].groupby(KEYS)[band].agg(['mean','count']).reset_index()
   pre=pre[pre['count']==2].rename(columns={'mean':'pre'})
   v=a.merge(pre[KEYS+['pre']],on=KEYS,validate='many_to_one');v['delta']=v[band]-v.pre
   # Require at least one valid observation in years +1,+2.
   post=v[v.rel.isin([1,2])].dropna(subset=['delta']).groupby(KEYS).delta.mean().rename('post_change').reset_index()
   post=post.merge(z,on=KEYS,validate='one_to_one')
   for s,g in post.groupby('site'):
    if len(g)>5 and g.post_change.nunique()>1 and g.cover_pp.nunique()>1:
     event_changes.append(dict(source=tag,threshold=th,band=band,site=s,n=len(g),rho_cover_vs_post_change=spearmanr(g.cover_pp,g.post_change).statistic,median_post_change=g.post_change.median()))
   v=v.merge(post[KEYS],on=KEYS,validate='many_to_one')
   for (s,rich,rel),g in v[v.rel.between(-3,6)].groupby(['site','rich','rel']):
    g=g.dropna(subset=['delta'])
    if len(g):event_rows.append(dict(source=tag,threshold=th,band=band,site=s,rich=rich,rel=rel,n=len(g),blocks=g.block21.nunique(),median=g.delta.median(),q25=g.delta.quantile(.25),q75=g.delta.quantile(.75)))
EV=save9(pd.DataFrame(event_rows),'event_aligned_HLS')
save9(pd.DataFrame(event_changes),'post_event_change_vs_abundance')
if len(EV):
 for tag in ['Hansen','JRC_first','JRC_last']:
  fig,axs=plt.subplots(2,2,figsize=(11,8),layout='constrained')
  for s,ax in zip(SITES,axs.ravel()):
   for rich,g in EV[(EV.source==tag)&EV.threshold.eq(.5)&EV.band.eq('NBR')&EV.site.eq(s)].groupby('rich'):
    g=g.sort_values('rel');ax.plot(g.rel,g['median'],'o-',label='≥40%' if rich else '<40%');ax.fill_between(g.rel,g.q25,g.q75,alpha=.15)
   ax.axvline(0,color='black',ls='--');ax.axhline(0,color='grey',lw=.7);ax.set(title=s,xlabel='Years relative to detected event',ylabel='NBR change from pre-event mean');ax.legend()
  fig9(fig,'event_aligned_NBR_'+tag)


## 06 — Integrative interpretation and reproducibility. Compare disturbance-only versus historical spectra versus contemporary mapping from completed Block 14C if available; no refitting. Outputs are evidence for ecological associations, not causal effects or independent product validation.

In [ ]:
for p in sorted((fp.parent/'validation').glob('*/equal_forest_summary.csv')):
 if (p.parent/'completion.json').is_file():
  print('Block 14 predictive context:',p.parent);display(pd.read_csv(p).round(3))
atomic_json(O9/'completion.json',dict(complete=True,reference_n=len(D9),history_n=len(Q),HLS_matched=int(D9.eligible14.sum()),outputs=sorted(p.name for p in O9.iterdir()),limitations=['four landscapes; no causal identification','Hansen no detected loss is not undisturbed','JRC and Hansen share Landsat inputs','event age is proxy; mixed cells and changing annual observation support','descriptive uncertainty conditional on study sites']))
print('NB09 COMPLETE',O9)
